In [12]:
from pathlib import Path

import geopandas as gpd


RAIZ_PROYECTO = Path.cwd()
RUTA_BASE = RAIZ_PROYECTO / "base" / "localidades_objetivo_sus.gpkg"
RUTA_SALIDA = RAIZ_PROYECTO / "public" / "mapa_base.geojson"

base = gpd.read_file(RUTA_BASE)

In [15]:
base = base[base['tipo_mge'] == 'ageb']

In [22]:
conteo_consul = base [ 'consultorios_faltantes'].unique()
conteo_consul

<FloatingArray>
[0.5, 0.2, 0.1, 0.7, 1.9, 0.3, 0.6, 1.1, 2.0, 1.0, 0.9, 1.3, 0.4, 0.0, 2.2,
 2.3, 2.8, 1.5, 1.2, 1.6, 0.8, 1.4, 2.1, 5.4, 2.6, 3.5, 3.0, 1.7, 1.8, 2.5,
 3.2, 3.1, 2.9, 3.8, 4.7, 2.4, 2.7]
Length: 37, dtype: Float64

In [21]:
base.columns

Index(['cve_loc', 'nom_loc', 'consultorios_faltantes', 'geometry'], dtype='object')

In [23]:


columnas_requeridas = {"cve_loc", "nom_loc", "consultorios_faltantes", "geometry"}
columnas_faltantes = columnas_requeridas.difference(base.columns)
if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas requeridas en la base: {sorted(columnas_faltantes)}"
    )

base = base[["cve_loc", "nom_loc", "consultorios_faltantes", "geometry"]].copy()
base["cve_loc"] = base["cve_loc"].astype("string").str.strip()
base["nom_loc"] = base["nom_loc"].astype("string").str.strip()
base["consultorios_faltantes"] = base["consultorios_faltantes"].astype("Float64")
base["consultorios_faltantes"] = base["consultorios_faltantes"].where(
    base["consultorios_faltantes"] > 0,
)
base = base.dropna(subset=["cve_loc", "nom_loc", "geometry"])
base = base[base.geometry.is_valid & ~base.geometry.is_empty]
base = base.drop_duplicates(subset=["cve_loc"], keep="first")

if base.crs is None:
    raise ValueError("La base no tiene CRS definido; no es seguro transformar geometry.")

base = base.to_crs("EPSG:6372")
base["geometry"] = base.geometry.simplify(100, preserve_topology=True)
base = base.to_crs("EPSG:4326")
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
base.to_file(RUTA_SALIDA, driver="GeoJSON")

print(f"Registros exportados: {len(base):,}")
print(f"CRS de salida: {base.crs}")
print(f"Archivo generado: {RUTA_SALIDA}")

Registros exportados: 1,128
CRS de salida: EPSG:4326
Archivo generado: c:\Users\jose.valdez\Downloads\nuevo-map\mapa-AGEB-\public\mapa_base.geojson
